# Phase A : prédire les prix immobiliers

In [9]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (r2_score,mean_absolute_error,mean_squared_error)

In [11]:
def charger_immobilier():
    
    data = fetch_california_housing()

    X = pd.DataFrame(
        data.data,
        columns=data.feature_names
    )

    y = pd.Series(
        data.target,
        name="Prix_median"
    )

    print("California Housing :", X.shape)
    print("Nombre de lignes :", X.shape[0])
    print("Nombre de variables :", X.shape[1])
    print("Variables :", list(X.columns))
    print("Unité de la cible : centaines de milliers de dollars")

    return X, y
X, y = charger_immobilier()

California Housing : (20640, 8)
Nombre de lignes : 20640
Nombre de variables : 8
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Unité de la cible : centaines de milliers de dollars


### voir les premiéres lignes de ma dataset

In [12]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


### Séparation entraînement/test

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (16512, 8)
X_test  : (4128, 8)
y_train : (16512,)
y_test  : (4128,)


### Standardisation des données

In [21]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train standardisé :", X_train_scaled.shape)
print("X_test standardisé  :", X_test_scaled.shape)

print("\nMoyennes après standardisation :")
print(np.round(X_train_scaled.mean(axis=0), 2))

print("\nÉcarts-types après standardisation :")
print(np.round(X_train_scaled.std(axis=0), 2))

X_train standardisé : (16512, 8)
X_test standardisé  : (4128, 8)

Moyennes après standardisation :
[-0. -0.  0. -0.  0.  0.  0.  0.]

Écarts-types après standardisation :
[1. 1. 1. 1. 1. 1. 1. 1.]


### Fonction d’évaluation

In [22]:
def evaluer_regression(modele, X_train_modele, X_test_modele, y_train, y_test):
    """
    Entraîne un modèle de régression et calcule
    les métriques R², MAE et RMSE.

    Retourne un dictionnaire contenant les trois métriques.
    """

    modele.fit(X_train_modele, y_train)

    predictions = modele.predict(X_test_modele)

    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    resultats = {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }

    print(f"R²    : {r2:.3f}")
    print(f"MAE   : {mae:.3f}")
    print(f"RMSE  : {rmse:.3f}")

    return resultats

#### Régression linéaire 

In [34]:
print("Résultats de la régression linéaire")

resultats_lineaire = evaluer_regression(
    regression_lineaire,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)



Résultats de la régression linéaire
R²    : 0.576
MAE   : 0.533
RMSE  : 0.746


#### Random Forest

In [35]:
random_forest = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
print(" Résultats du Random Forest")

resultats_forest = evaluer_regression(
    random_forest,
    X_train,
    X_test,
    y_train,
    y_test
)



 Résultats du Random Forest
R²    : 0.806
MAE   : 0.327
RMSE  : 0.504


### Comparaison des modèles 

In [39]:
print(
    f"California Housing : {X.shape} variables, "
    "cible = prix médian en centaines de milliers de $"
)

print(
    f"LinearRegression : "
    f"R2={resultats_lineaire['R2']:.2f} "
    f"MAE={resultats_lineaire['MAE']:.2f} "
    f"RMSE={resultats_lineaire['RMSE']:.2f}"
)

print(
    f"RandomForest : "
    f"R2={resultats_forest['R2']:.2f} "
    f"MAE={resultats_forest['MAE']:.2f} "
    f"RMSE={resultats_forest['RMSE']:.2f}"
)

California Housing : (20640, 8) variables, cible = prix médian en centaines de milliers de $
LinearRegression : R2=0.58 MAE=0.53 RMSE=0.75
RandomForest : R2=0.81 MAE=0.33 RMSE=0.50


### identification du meilleur modèle

In [30]:
meilleur_modele = comparaison_modeles["R2"].idxmax()

print("Modèle ayant le meilleur R² :", meilleur_modele)


Modèle ayant le meilleur R² : Random Forest


#### Justification : 


Les deux modèles ont été entraînés et évalués sur le même découpage des données.


le meilleur modéle étiat le  Random Forest car il obtient  un R² plus élevé ainsi qu’une MAE et une RMSE plus faibles. Il est donc généralement plus performant sur ce dataset.

Cela indique que la relation entre les caractéristiques des quartiers et le prix des logements n’est probablement pas uniquement linéaire.

#### Test 1 : Cas normal

##### Regression linéaire 

In [42]:
print("Régression linéaire : dataset complet ")

resultats_lineaire = evaluer_regression(
    regression_lineaire,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)


Régression linéaire : dataset complet 
R²    : 0.576
MAE   : 0.533
RMSE  : 0.746


##### Randomforest 

In [43]:
print(" Random Forest : dataset complet ")

resultats_forest = evaluer_regression(
    random_forest,
    X_train,
    X_test,
    y_train,
    y_test
)



 Random Forest : dataset complet 
R²    : 0.806
MAE   : 0.327
RMSE  : 0.504


#### Test 2 : prendre 100 lignes 

In [44]:
X_100 = X.iloc[:100]
y_100 = y.iloc[:100]

print("Taille du dataset limité :", X_100.shape)

Taille du dataset limité : (100, 8)


#### Découpage train/test et Standardisation

In [48]:
X_train_100, X_test_100, y_train_100, y_test_100 = train_test_split(
    X_100,
    y_100,
    test_size=0.20,
    random_state=42
)

scaler_100 = StandardScaler()

X_train_100_scaled = scaler_100.fit_transform(X_train_100)
X_test_100_scaled = scaler_100.transform(X_test_100)



#### régression linéaire

In [49]:
regression_lineaire_100 = LinearRegression()
print(" Régression linéaire avec 100 lignes ")

resultats_lineaire_100 = evaluer_regression(
    regression_lineaire_100,
    X_train_100_scaled,
    X_test_100_scaled,
    y_train_100,
    y_test_100
)


 Régression linéaire avec 100 lignes 
R²    : 0.712
MAE   : 0.384
RMSE  : 0.493


#### Random Forest

In [50]:
random_forest_100 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
print(" Random Forest avec 100 lignes ")

resultats_forest_100 = evaluer_regression(
    random_forest_100,
    X_train_100,
    X_test_100,
    y_train_100,
    y_test_100
)


 Random Forest avec 100 lignes 
R²    : 0.812
MAE   : 0.264
RMSE  : 0.398


#### comparaison des deux modeles : 

In [51]:
comparaison_100 = pd.DataFrame({
    "Régression linéaire": resultats_lineaire_100,
    "Random Forest": resultats_forest_100
}).T

print("Résultats avec seulement 100 lignes :")

display(comparaison_100.round(3))

Résultats avec seulement 100 lignes :


,R2,MAE,RMSE
Régression linéaire,0.712,0.384,0.493
Random Forest,0.812,0.264,0.398


#### Cas adversarial

In [53]:
print("Plage observée pour MedInc :")
print("Minimum :", X["MedInc"].min())
print("Maximum :", X["MedInc"].max())

print("\nPlage observée pour Population :")
print("Minimum :", X["Population"].min())
print("Maximum :", X["Population"].max())

Plage observée pour MedInc :
Minimum : 0.4999
Maximum : 15.0001

Plage observée pour Population :
Minimum : 3.0
Maximum : 35682.0


#### Création du quartier fictif

In [54]:
quartier_fictif = X.median().to_frame().T

quartier_fictif["MedInc"] = 0
quartier_fictif["Population"] = 9000

print("Quartier fictif utilisé :")

display(quartier_fictif)

Quartier fictif utilisé :


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,0,29.0,5.229129,1.04878,9000,2.818116,34.26,-118.49


#### Standardisation pour la régression linéaire

In [57]:
quartier_fictif_scaled = scaler.transform(quartier_fictif)
prediction_lineaire = regression_lineaire.predict(
    quartier_fictif_scaled
)[0]

print("Régression linéaire ")
print(
    "Prédiction brute :",
    round(prediction_lineaire, 3),
    "centaines de milliers de dollars"
)

print(
    "Prix estimé : environ",
    round(prediction_lineaire * 100_000),
    "dollars"
)


Régression linéaire 
Prédiction brute : 0.415 centaines de milliers de dollars
Prix estimé : environ 41497 dollars


#### RandomForest 

In [58]:
prediction_forest = random_forest.predict(
    quartier_fictif
)[0]

print("Random Forest ")
print(
    "Prédiction brute :",
    round(prediction_forest, 3),
    "centaines de milliers de dollars"
)

print(
    "Prix estimé : environ",
    round(prediction_forest * 100_000),
    "dollars"
)

Random Forest 
Prédiction brute : 1.959 centaines de milliers de dollars
Prix estimé : environ 195882 dollars
